In [ ]:
import os

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [ ]:
ROOT = ".."
DATA_PATH = os.path.join(ROOT, "data", "inference.parquet")
BASE_MODEL_NAME = "DeepPavlov-rubert-base-cased"
BASE_MODEL_PATH = os.path.join(ROOT, "models", BASE_MODEL_NAME)

TEXT_COL = "text"
TARGET_COL = "target"

FINETUNED_MODEL_NAME = f"{BASE_MODEL_NAME}-injection-finetuned"
FINETUNED_MODEL_PATH = os.path.join(ROOT, "models", FINETUNED_MODEL_NAME) # куда сохранится дообученная модель

MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
inference_df = pd.read_parquet(DATA_PATH, columns=["text_ru", "label", "filter"]).rename(columns={"text_ru": "text"})
inference_df = inference_df[inference_df["filter"] != 1].drop(columns=["filter"]).reset_index(drop=True)
inference_df["target"] = inference_df["label"].replace({"injection": 1, "benign": 0}).astype("int8")

print(inference_df.shape)
inference_df.head()

In [ ]:
train_df, val_df = train_test_split(
    inference_df, test_size=0.2, random_state=42, stratify=inference_df[TARGET_COL]
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"train: {train_df.shape} | val: {val_df.shape}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_PATH, local_files_only=True, num_labels=2
)

if tokenizer.pad_token is None:
    pad_id = getattr(model.config, "pad_token_id", None)
    if pad_id is not None and pad_id < len(tokenizer):
        tokenizer.pad_token = tokenizer.convert_ids_to_tokens(pad_id)
    else:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})
        model.resize_token_embeddings(len(tokenizer))
    model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True, max_length=max_length
        )
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
train_dataset = TextClassificationDataset(train_df[TEXT_COL], train_df[TARGET_COL], tokenizer, MAX_LENGTH)
val_dataset = TextClassificationDataset(val_df[TEXT_COL], val_df[TARGET_COL], tokenizer, MAX_LENGTH)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=os.path.join(ROOT, "train_runs", FINETUNED_MODEL_NAME),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=20,
    report_to=[],
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()
print(trainer.evaluate())

In [ ]:
os.makedirs(FINETUNED_MODEL_PATH, exist_ok=True)
trainer.save_model(FINETUNED_MODEL_PATH)
tokenizer.save_pretrained(FINETUNED_MODEL_PATH)
print(f"Дообученная модель сохранена: {FINETUNED_MODEL_PATH}")